<a href="https://colab.research.google.com/github/Sarunwit00/Tensorflow/blob/main/VoiceTest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install audiomentations librosa soundfile

In [10]:
import librosa
import soundfile as sf
import numpy as np
from audiomentations import Compose, PitchShift, TimeStretch, AddGaussianNoise
import os
import random

# 1. ตั้งค่าโฟลเดอร์
input_folder = '/content/Input'
output_folder = '/content/Output'

if not os.path.exists(input_folder):
    os.makedirs(input_folder, exist_ok=True)
    print(f"สร้างโฟลเดอร์อินพุตเรียบร้อยแล้วที่: {input_folder}")

if not os.path.exists(output_folder):
    os.makedirs(output_folder, exist_ok=True)

# 2. กำหนดรูปแบบการเปลี่ยนเสียง (Voice Profiles)
# เสียงเด็ก (Pitch สูงมาก, พูดเร็วขึ้นนิดหน่อย)
aug_child = Compose([
    PitchShift(min_semitones=5, max_semitones=8, p=1.0),
    TimeStretch(min_rate=1.0, max_rate=1.2, p=0.5)
])

# เสียงผู้หญิง (Pitch สูงขึ้นปานกลาง)
aug_female = Compose([
    PitchShift(min_semitones=2, max_semitones=5, p=1.0)
])

# เสียงคนแก่ (Pitch ต่ำลงนิดหน่อย, พูดช้าลง)
aug_elderly = Compose([
    PitchShift(min_semitones=-3, max_semitones=-1, p=1.0),
    TimeStretch(min_rate=0.75, max_rate=0.9, p=1.0)
])

# เสียงผู้ชายทุ้มลึก (Pitch ต่ำมาก)
aug_deep_male = Compose([
    PitchShift(min_semitones=-6, max_semitones=-3, p=1.0)
])

# เสียงแบบมี Noise หรือสภาพแวดล้อมอื่นๆ
aug_noisy_other = Compose([
    PitchShift(min_semitones=-2, max_semitones=2, p=0.8),
    AddGaussianNoise(min_amplitude=0.005, max_amplitude=0.02, p=1.0)
])

# รวมชุด Augmentation ไว้สุ่มเลือก
augmenters = [aug_child, aug_female, aug_elderly, aug_deep_male, aug_noisy_other]

# 3. เริ่มประมวลผล
sample_rate = 16000 # ต้องเป็น 16,000 Hz ตามมาตรฐานโมเดล

files = [f for f in os.listdir(input_folder) if f.endswith('.wav')]

if len(files) > 0:
    print(f"กำลังเริ่มประมวลผลไฟล์จำนวน {len(files)} ไฟล์...")
    for filename in files:
        path = os.path.join(input_folder, filename)
        # โหลดไฟล์เสียงให้เป็น Mono (ช่องเสียงเดียว)
        audio, sr = librosa.load(path, sr=sample_rate, mono=True)

        print(f"กำลังสร้าง 100 เสียงใหม่จากไฟล์: {filename}")
        # ลูปสร้าง 100 ไฟล์ต่อ 1 ไฟล์ต้นฉบับ
        for i in range(100):
            # สุ่มเลือกว่าจะใช้รูปแบบเสียงไหน (เด็ก, ผู้หญิง, คนแก่ ฯลฯ)
            selected_aug = random.choice(augmenters)

            # ทำการแปลงเสียง
            augmented_audio = selected_aug(samples=audio, sample_rate=sample_rate)

            # บันทึกไฟล์
            new_filename = f"{filename.split('.')[0]}_aug_{i+1}.wav"
            output_path = os.path.join(output_folder, new_filename)
            sf.write(output_path, augmented_audio, sample_rate)

    print(f"เสร็จเรียบร้อย! ไฟล์ที่แปลงแล้วทั้งหมดอยู่ที่: {output_folder}")
else:
    print("ไม่พบไฟล์ .wav ในโฟลเดอร์อินพุต กรุณาตรวจสอบไฟล์ของคุณ")

กำลังเริ่มประมวลผลไฟล์จำนวน 1 ไฟล์...
กำลังสร้าง 100 เสียงใหม่จากไฟล์: voice_1_1.wav
เสร็จเรียบร้อย! ไฟล์ที่แปลงแล้วทั้งหมดอยู่ที่: /content/Output


In [11]:
import os
import shutil
from google.colab import files as colab_files
import ipywidgets as widgets
from IPython.display import display

def download_all_files(b):
    output_zip = '/content/augmented_audio_results'
    if os.path.exists(output_folder) and len(os.listdir(output_folder)) > 0:
        print("กำลังบีบอัดไฟล์... กรุณารอสักครู่")
        shutil.make_archive(output_zip, 'zip', output_folder)
        # ใช้ colab_files เพื่อป้องกันชื่อซ้ำกับตัวแปร list
        colab_files.download(f"{output_zip}.zip")
        print("เริ่มการดาวน์โหลดเรียบร้อยแล้ว!")
    else:
        print("ไม่พบไฟล์ในโฟลเดอร์ Output")

def clear_all_files(b):
    if os.path.exists(output_folder):
        for filename in os.listdir(output_folder):
            file_path = os.path.join(output_folder, filename)
            try:
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.unlink(file_path)
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)
            except Exception as e:
                print(f'ไม่สามารถลบ {file_path} ได้ เนื่องจาก: {e}')
        print("ลบไฟล์ทั้งหมดในโฟลเดอร์ Output เรียบร้อยแล้ว!")

# สร้างปุ่ม
btn_download = widgets.Button(description="Download All (.zip)", button_style='success', layout=widgets.Layout(width='200px'))
btn_clear = widgets.Button(description="Clear Output Folder", button_style='danger', layout=widgets.Layout(width='200px'))

btn_download.on_click(download_all_files)
btn_clear.on_click(clear_all_files)

display(widgets.HBox([btn_download, btn_clear]))

กำลังบีบอัดไฟล์... กรุณารอสักครู่


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

เริ่มการดาวน์โหลดเรียบร้อยแล้ว!
ลบไฟล์ทั้งหมดในโฟลเดอร์ Output เรียบร้อยแล้ว!
